# Capítulo 1: Estrutura Básica de Prompt

- [Lição](#lesson)
- [Exercícios](#exercises)
- [Playground de Exemplos](#example-playground)

## Configuração

Execute a célula de configuração abaixo para carregar sua chave de API e estabelecer a função auxiliar `get_completion`.

In [ ]:
import sys, re
sys.path.insert(0, '/workspaces/prompt-eng-ia')
from ollama_helper import get_completion, MODEL_NAME, client

---

## Lição

A Anthropic oferece duas APIs: a legada [API de Completions de Texto](https://docs.anthropic.com/claude/reference/complete_post) e a atual [API de Mensagens](https://docs.anthropic.com/claude/reference/messages_post). Para este tutorial, usaremos exclusivamente a API de Mensagens.

No mínimo, uma chamada ao Claude usando a API de Mensagens requer os seguintes parâmetros:
- `model`: o [nome do modelo na API](https://docs.anthropic.com/claude/docs/models-overview#model-recommendations) do modelo que você pretende chamar

- `max_tokens`: o número máximo de tokens a gerar antes de parar. Note que o Claude pode parar antes de atingir esse máximo. Este parâmetro especifica apenas o número absoluto máximo de tokens a gerar. Além disso, este é um limite *rígido*, o que significa que pode fazer o Claude parar de gerar no meio de uma palavra ou frase.

- `messages`: um array de mensagens de entrada. Nossos modelos são treinados para operar em turnos conversacionais alternados entre `user` e `assistant`. Ao criar uma nova `Message`, você especifica os turnos conversacionais anteriores com o parâmetro messages, e o modelo então gera a próxima `Message` na conversa.
  - Cada mensagem de entrada deve ser um objeto com `role` e `content`. Você pode especificar uma única mensagem com o papel `user`, ou incluir múltiplas mensagens de `user` e `assistant` (elas devem se alternar, nesse caso). A primeira mensagem deve sempre usar o papel `user`.

Há também parâmetros opcionais, como:
- `system`: o system prompt - mais sobre isso abaixo.
  
- `temperature`: o grau de variabilidade na resposta do Claude. Para estas lições e exercícios, definimos `temperature` como 0.

Para uma lista completa de todos os parâmetros da API, visite nossa [documentação da API](https://docs.anthropic.com/claude/reference/messages_post).

### Exemplos

Vamos ver como o Claude responde a alguns prompts formatados corretamente. Para cada uma das células a seguir, execute a célula (`shift+enter`), e a resposta do Claude aparecerá abaixo do bloco.

In [ ]:
# Prompt
PROMPT = "Hi Claude, how are you?"

# Imprime a resposta do Claude
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Can you tell me the color of the ocean?"

# Imprime a resposta do Claude
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "What year was Celine Dion born in?"

# Imprime a resposta do Claude
print(get_completion(PROMPT))

Agora vamos ver alguns prompts que não incluem a formatação correta da API de Mensagens. Para esses prompts mal formatados, a API de Mensagens retorna um erro.

Primeiro, temos um exemplo de uma chamada à API de Mensagens que não possui os campos `role` e `content` no array `messages`.

In [ ]:
# Obtém a resposta do Claude
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"Hi Claude, how are you?"}
        ]
    )

# Imprime a resposta do Claude
print(response[0].text)

Aqui está um prompt que não alterna entre os papéis `user` e `assistant`.

In [ ]:
# Obtém a resposta do Claude
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": "What year was Celine Dion born in?"},
          {"role": "user", "content": "Also, can you tell me some other facts about her?"}
        ]
    )

# Imprime a resposta do Claude
print(response[0].text)

As mensagens de `user` e `assistant` **DEVEM se alternar**, e as mensagens **DEVEM começar com um turno de `user`**. Você pode ter múltiplos pares de `user` e `assistant` em um prompt (como se estivesse simulando uma conversa de múltiplos turnos). Você também pode colocar palavras em uma mensagem terminal de `assistant` para que o Claude continue a partir de onde parou (mais sobre isso em capítulos posteriores).

#### System Prompts

Você também pode usar **system prompts**. Um system prompt é uma forma de **fornecer contexto, instruções e diretrizes ao Claude** antes de apresentar a ele uma pergunta ou tarefa no turno de "User". 

Estruturalmente, os system prompts existem separadamente da lista de mensagens de `user` e `assistant`, e portanto pertencem a um parâmetro `system` separado (veja a estrutura da função auxiliar `get_completion` na seção [Configuração](#setup) do notebook). 

Neste tutorial, onde quer que possamos utilizar um system prompt, fornecemos um campo `system` na sua função de completions. Caso não queira usar um system prompt, simplesmente defina a variável `SYSTEM_PROMPT` como uma string vazia.

#### Exemplo de System Prompt

In [ ]:
# System prompt
SYSTEM_PROMPT = "Your answer should always be a series of critical thinking questions that further the conversation (do not provide answers to your questions). Do not actually answer the user question."

# Prompt
PROMPT = "Why is the sky blue?"

# Imprime a resposta do Claude
print(get_completion(PROMPT, SYSTEM_PROMPT))

Por que usar um system prompt? Um **system prompt bem escrito pode melhorar o desempenho do Claude** de diversas formas, como aumentar a capacidade do Claude de seguir regras e instruções. Para mais informações, visite nossa documentação sobre [como usar system prompts](https://docs.anthropic.com/claude/docs/how-to-use-system-prompts) com o Claude.

Agora vamos mergulhar em alguns exercícios. Se quiser experimentar os prompts da lição sem alterar nenhum conteúdo acima, role até o final do notebook da lição para visitar o [**Playground de Exemplos**](#example-playground).

---

## Exercícios
- [Exercício 1.1 - Contando até Três](#exercise-11---counting-to-three)
- [Exercício 1.2 - System Prompt](#exercise-12---system-prompt)

### Exercício 1.1 - Contando até Três
Usando a formatação correta de `user` / `assistant`, edite o `PROMPT` abaixo para fazer o Claude **contar até três.** A saída também indicará se sua solução está correta.

In [ ]:
# Prompt - este é o único campo que você deve alterar
PROMPT = "[Replace this text]"

# Obtém a resposta do Claude
response = get_completion(PROMPT)

# Função para avaliar a correção do exercício
def grade_exercise(text):
    pattern = re.compile(r'^(?=.*1)(?=.*2)(?=.*3).*$', re.DOTALL)
    return bool(pattern.match(text))

# Imprime a resposta do Claude e a nota correspondente
print(response)
print("\n--------------------------- AVALIAÇÃO ---------------------------")
print("Este exercício foi resolvido corretamente:", grade_exercise(response))

❓ Se quiser uma dica, execute a célula abaixo!

In [ ]:
from hints import exercise_1_1_hint; print(exercise_1_1_hint)

### Exercício 1.2 - System Prompt

Modifique o `SYSTEM_PROMPT` para fazer o Claude responder como se fosse uma criança de 3 anos.

In [ ]:
# System prompt - este é o único campo que você deve alterar
SYSTEM_PROMPT = "[Replace this text]"

# Prompt
PROMPT = "How big is the sky?"

# Obtém a resposta do Claude
response = get_completion(PROMPT, SYSTEM_PROMPT)

# Função para avaliar a correção do exercício
def grade_exercise(text):
    return bool(re.search(r"giggles", text) or re.search(r"soo", text))

# Imprime a resposta do Claude e a nota correspondente
print(response)
print("\n--------------------------- AVALIAÇÃO ---------------------------")
print("Este exercício foi resolvido corretamente:", grade_exercise(response))

❓ Se quiser uma dica, execute a célula abaixo!

In [ ]:
from hints import exercise_1_2_hint; print(exercise_1_2_hint)

### Parabéns!

Se você resolveu todos os exercícios até este ponto, está pronto para passar para o próximo capítulo. Bons prompts!

---

## Playground de Exemplos

Esta é uma área para você experimentar livremente com os exemplos de prompts mostrados nesta lição e ajustar os prompts para ver como isso pode afetar as respostas do Claude.

In [ ]:
# Prompt
PROMPT = "Hi Claude, how are you?"

# Imprime a resposta do Claude
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Can you tell me the color of the ocean?"

# Imprime a resposta do Claude
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "What year was Celine Dion born in?"

# Imprime a resposta do Claude
print(get_completion(PROMPT))

In [ ]:
# Obtém a resposta do Claude
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"Hi Claude, how are you?"}
        ]
    )

# Imprime a resposta do Claude
print(response[0].text)

In [ ]:
# Obtém a resposta do Claude
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": "What year was Celine Dion born in?"},
          {"role": "user", "content": "Also, can you tell me some other facts about her?"}
        ]
    )

# Imprime a resposta do Claude
print(response[0].text)

In [ ]:
# System prompt
SYSTEM_PROMPT = "Your answer should always be a series of critical thinking questions that further the conversation (do not provide answers to your questions). Do not actually answer the user question."

# Prompt
PROMPT = "Why is the sky blue?"

# Imprime a resposta do Claude
print(get_completion(PROMPT, SYSTEM_PROMPT))